In [1]:
import lightgbm as lgb
print(lgb.__version__)

4.7.0


In [2]:
# Step 2: Load saved artifacts and recreate train/validation data

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import sklearn
import lightgbm as lgb

from scipy import sparse

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

# 1. Recreate RareCategoryGrouper before loading preprocessor

class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    """
    Converts categorical values to strings and groups categories
    occurring fewer than min_frequency times into __RARE__.
    """

    def __init__(self, min_frequency=10, rare_label="__RARE__"):
        self.min_frequency = min_frequency
        self.rare_label = rare_label

    def fit(self, X, y=None):
        X_array = np.asarray(X, dtype=object).astype(str)

        if X_array.ndim != 2:
            raise ValueError(
                "RareCategoryGrouper expects a two-dimensional input."
            )

        self.n_features_in_ = X_array.shape[1]
        self.frequent_categories_ = []

        for column_index in range(self.n_features_in_):
            category_counts = pd.Series(
                X_array[:, column_index]
            ).value_counts(dropna=False)

            frequent_categories = set(
                category_counts[
                    category_counts >= self.min_frequency
                ].index.tolist()
            )

            self.frequent_categories_.append(frequent_categories)

        return self

    def transform(self, X):
        check_is_fitted(self, attributes=["frequent_categories_"])

        X_array = np.asarray(X, dtype=object).astype(str)

        if X_array.ndim != 2:
            raise ValueError(
                "RareCategoryGrouper expects a two-dimensional input."
            )

        if X_array.shape[1] != self.n_features_in_:
            raise ValueError(
                "The number of categorical columns differs from the fitted data."
            )

        for column_index, frequent_categories in enumerate(
            self.frequent_categories_
        ):
            column_values = X_array[:, column_index]

            frequent_mask = np.fromiter(
                (
                    value in frequent_categories
                    for value in column_values
                ),
                dtype=bool,
                count=len(column_values)
            )

            column_values[~frequent_mask] = self.rare_label
            X_array[:, column_index] = column_values

        return X_array

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return np.asarray(
                [
                    f"x{index}"
                    for index in range(self.n_features_in_)
                ],
                dtype=object
            )

        return np.asarray(input_features, dtype=object)

# 2. Detect project root

CURRENT_DIRECTORY = Path.cwd().resolve()

PROJECT_ROOT = None

for candidate in [CURRENT_DIRECTORY, *CURRENT_DIRECTORY.parents]:
    expected_file = (
        candidate
        / "data"
        / "processed"
        / "diabetic_modeling_data_final.csv"
    )

    if expected_file.exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate project root."
    )

print("Current working directory:", CURRENT_DIRECTORY)
print("Project root detected:", PROJECT_ROOT)

# 3. Define required paths

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "diabetic_modeling_data_final.csv"
)

SCHEMA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "model_feature_schema.json"
)

SPLIT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "patient_split_assignments.csv"
)

FITTED_PREPROCESSOR_PATH = (
    PROJECT_ROOT
    / "models"
    / "notebook_4_fitted_preprocessor.joblib"
)

NOTEBOOK_6_METADATA_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "notebook_6_model_tuning_threshold_metadata.json"
)

NOTEBOOK_6_COMPARISON_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "metrics"
    / "notebook_6_tuned_model_comparison.csv"
)

required_files = {
    "Final modeling dataset": DATA_PATH,
    "Feature schema": SCHEMA_PATH,
    "Patient split assignments": SPLIT_PATH,
    "Fitted preprocessor": FITTED_PREPROCESSOR_PATH,
    "Notebook 6 metadata": NOTEBOOK_6_METADATA_PATH,
    "Notebook 6 tuned comparison": NOTEBOOK_6_COMPARISON_PATH,
}

missing_files = [
    f"{name}: {path}"
    for name, path in required_files.items()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Missing required files:\n"
        + "\n".join(missing_files)
    )

print("\nAll required Notebook 6B input files were found.")

# 4. Load saved artifacts

df = pd.read_csv(DATA_PATH)
split_assignments = pd.read_csv(SPLIT_PATH)

with open(SCHEMA_PATH, "r", encoding="utf-8") as file:
    feature_schema = json.load(file)

with open(NOTEBOOK_6_METADATA_PATH, "r", encoding="utf-8") as file:
    notebook_6_metadata = json.load(file)

notebook_6_comparison = pd.read_csv(NOTEBOOK_6_COMPARISON_PATH)

fitted_preprocessor = joblib.load(FITTED_PREPROCESSOR_PATH)

# 5. Read feature contract

TARGET_COLUMN = feature_schema["target_column"]
GROUP_COLUMN = feature_schema["group_split_column"]

NUMERIC_FEATURES = feature_schema["numeric_model_features"]
CATEGORICAL_FEATURES = feature_schema["categorical_model_features"]
MODEL_FEATURES = feature_schema["all_model_features"]

# 6. Recreate train, validation, and reserved test partitions

modeling_df = df.merge(
    split_assignments[["encounter_id", "split"]],
    on="encounter_id",
    how="left",
    validate="one_to_one"
)

if modeling_df["split"].isna().any():
    raise ValueError(
        "Some encounters did not receive a split assignment."
    )

train_df = modeling_df[
    modeling_df["split"] == "train"
].copy()

validation_df = modeling_df[
    modeling_df["split"] == "validation"
].copy()

test_df = modeling_df[
    modeling_df["split"] == "test"
].copy()

X_train = train_df[MODEL_FEATURES].copy()
y_train = train_df[TARGET_COLUMN].copy()

X_validation = validation_df[MODEL_FEATURES].copy()
y_validation = validation_df[TARGET_COLUMN].copy()

# Important:
# Do not create X_test or y_test in Notebook 6B.
# Test set remains untouched.

# 7. Transform train and validation using saved preprocessor

X_train_preprocessed = fitted_preprocessor.transform(X_train)
X_validation_preprocessed = fitted_preprocessor.transform(X_validation)

# 8. Define evaluation function

def evaluate_binary_classifier(
    model_name,
    y_true,
    y_pred,
    y_probability
):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )

    predicted_positive_rate = np.mean(y_pred == 1)

    results = {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            y_pred
        ),
        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "recall_sensitivity": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "specificity": specificity,
        "f1_score": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "roc_auc": roc_auc_score(
            y_true,
            y_probability
        ),
        "pr_auc": average_precision_score(
            y_true,
            y_probability
        ),
        "predicted_positive_rate": predicted_positive_rate,
        "true_negatives": tn,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp,
    }

    return pd.DataFrame([results])

# 9. Display setup confirmation

print("\nNotebook 6B Step 2 completed successfully.")
print("-" * 75)

print(f"Pandas version: {pd.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")
print(f"LightGBM version: {lgb.__version__}")

print("\nDataset:")
print(f"Shape: {df.shape}")
print(f"Missing values: {df.isna().sum().sum():,}")

print("\nFeature contract:")
print(f"Target column: {TARGET_COLUMN}")
print(f"Patient grouping column: {GROUP_COLUMN}")
print(f"Numeric predictors: {len(NUMERIC_FEATURES)}")
print(f"Categorical predictors: {len(CATEGORICAL_FEATURES)}")
print(f"Total raw predictors: {len(MODEL_FEATURES)}")

print("\nTraining partition:")
print(f"Rows: {len(train_df):,}")
print(f"Patients: {train_df[GROUP_COLUMN].nunique():,}")
print(f"Positive rate: {y_train.mean():.3%}")
print(f"Preprocessed shape: {X_train_preprocessed.shape}")

print("\nValidation partition:")
print(f"Rows: {len(validation_df):,}")
print(f"Patients: {validation_df[GROUP_COLUMN].nunique():,}")
print(f"Positive rate: {y_validation.mean():.3%}")
print(f"Preprocessed shape: {X_validation_preprocessed.shape}")

print("\nReserved test partition:")
print(f"Rows reserved: {len(test_df):,}")
print(f"Patients reserved: {test_df[GROUP_COLUMN].nunique():,}")
print("Test predictors and labels were not created.")

print("\nNotebook 6 selected candidate:")
print(
    notebook_6_metadata["selected_candidate"]["selected_model_name"]
)
print(
    "Selected threshold:",
    notebook_6_metadata["selected_candidate"]["selected_threshold"]
)

print("\nNotebook 6B rules:")
print("- LightGBM will be tested as an optional advanced model.")
print("- Soft voting ensemble will be tested after LightGBM.")
print("- Test set remains untouched.")

Current working directory: C:\Users\pradh\Documents\hospital-readmission-project\notebooks
Project root detected: C:\Users\pradh\Documents\hospital-readmission-project

All required Notebook 6B input files were found.

Notebook 6B Step 2 completed successfully.
---------------------------------------------------------------------------
Pandas version: 3.0.3
Scikit-learn version: 1.9.0
LightGBM version: 4.7.0

Dataset:
Shape: (99343, 46)
Missing values: 0

Feature contract:
Target column: readmitted_30
Patient grouping column: patient_nbr
Numeric predictors: 8
Categorical predictors: 35
Total raw predictors: 43

Training partition:
Rows: 69,467
Patients: 48,993
Positive rate: 11.424%
Preprocessed shape: (69467, 179)

Validation partition:
Rows: 14,900
Patients: 10,498
Positive rate: 11.275%
Preprocessed shape: (14900, 179)

Reserved test partition:
Rows reserved: 14,976
Patients reserved: 10,499
Test predictors and labels were not created.

Notebook 6 selected candidate:
Best Tuned XGBo

In [3]:
# Step 3: Train and evaluate LightGBM candidate models

import time
import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier

# 1. Calculate class imbalance weight using training data only

negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())

base_scale_pos_weight = negative_count / positive_count

print("Training class distribution:")
print(f"Negative class count: {negative_count:,}")
print(f"Positive class count: {positive_count:,}")
print(f"Base scale_pos_weight: {base_scale_pos_weight:.4f}")

# 2. Define LightGBM tuning configurations

lightgbm_tuning_configs = [
    {
        "config_name": "LGBM_01_Balanced_Reference",
        "n_estimators": 300,
        "learning_rate": 0.05,
        "num_leaves": 31,
        "max_depth": 4,
        "min_child_samples": 50,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "reg_lambda": 1.0,
        "reg_alpha": 0.0,
        "scale_pos_weight_multiplier": 1.00,
    },
    {
        "config_name": "LGBM_02_More_Regularized",
        "n_estimators": 400,
        "learning_rate": 0.03,
        "num_leaves": 31,
        "max_depth": 4,
        "min_child_samples": 75,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_lambda": 3.0,
        "reg_alpha": 0.1,
        "scale_pos_weight_multiplier": 1.00,
    },
    {
        "config_name": "LGBM_03_Deeper_Trees",
        "n_estimators": 300,
        "learning_rate": 0.05,
        "num_leaves": 63,
        "max_depth": 5,
        "min_child_samples": 75,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "reg_lambda": 2.0,
        "reg_alpha": 0.1,
        "scale_pos_weight_multiplier": 1.00,
    },
    {
        "config_name": "LGBM_04_Lower_Positive_Weight",
        "n_estimators": 350,
        "learning_rate": 0.04,
        "num_leaves": 31,
        "max_depth": 4,
        "min_child_samples": 60,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_lambda": 2.0,
        "reg_alpha": 0.1,
        "scale_pos_weight_multiplier": 0.85,
    },
    {
        "config_name": "LGBM_05_Higher_Positive_Weight",
        "n_estimators": 350,
        "learning_rate": 0.04,
        "num_leaves": 31,
        "max_depth": 4,
        "min_child_samples": 60,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_lambda": 2.0,
        "reg_alpha": 0.1,
        "scale_pos_weight_multiplier": 1.15,
    },
]

# 3. Train and evaluate each LightGBM configuration

lightgbm_tuning_results = []
lightgbm_tuned_models = {}

for config in lightgbm_tuning_configs:

    config_name = config["config_name"]

    scale_pos_weight = (
        base_scale_pos_weight
        * config["scale_pos_weight_multiplier"]
    )

    print(f"\nTraining {config_name}...")
    print(f"scale_pos_weight: {scale_pos_weight:.4f}")

    model = LGBMClassifier(
        objective="binary",
        boosting_type="gbdt",
        n_estimators=config["n_estimators"],
        learning_rate=config["learning_rate"],
        num_leaves=config["num_leaves"],
        max_depth=config["max_depth"],
        min_child_samples=config["min_child_samples"],
        subsample=config["subsample"],
        colsample_bytree=config["colsample_bytree"],
        reg_lambda=config["reg_lambda"],
        reg_alpha=config["reg_alpha"],
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
        force_row_wise=True
    )

    start_time = time.perf_counter()

    model.fit(
        X_train_preprocessed,
        y_train
    )

    training_time = time.perf_counter() - start_time

    validation_probabilities = (
        model.predict_proba(
            X_validation_preprocessed
        )[:, 1]
    )

    validation_predictions = (
        validation_probabilities >= 0.50
    ).astype(int)

    result = evaluate_binary_classifier(
        model_name=config_name,
        y_true=y_validation,
        y_pred=validation_predictions,
        y_probability=validation_probabilities
    )

    result["training_time_seconds"] = training_time
    result["n_estimators"] = config["n_estimators"]
    result["learning_rate"] = config["learning_rate"]
    result["num_leaves"] = config["num_leaves"]
    result["max_depth"] = config["max_depth"]
    result["min_child_samples"] = config["min_child_samples"]
    result["subsample"] = config["subsample"]
    result["colsample_bytree"] = config["colsample_bytree"]
    result["reg_lambda"] = config["reg_lambda"]
    result["reg_alpha"] = config["reg_alpha"]
    result["scale_pos_weight"] = scale_pos_weight
    result["scale_pos_weight_multiplier"] = (
        config["scale_pos_weight_multiplier"]
    )

    lightgbm_tuning_results.append(result)
    lightgbm_tuned_models[config_name] = model

    print(
        f"Completed {config_name} in "
        f"{training_time:.2f} seconds."
    )

# 4. Combine LightGBM tuning results

lightgbm_tuning_results = pd.concat(
    lightgbm_tuning_results,
    ignore_index=True
)

lightgbm_tuning_results = (
    lightgbm_tuning_results
    .sort_values(
        by=[
            "balanced_accuracy",
            "f1_score",
            "roc_auc",
            "pr_auc",
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

# 5. Select best LightGBM model

best_lightgbm_row = lightgbm_tuning_results.iloc[0]
best_lightgbm_config_name = best_lightgbm_row["model"]

best_lightgbm_model = lightgbm_tuned_models[
    best_lightgbm_config_name
]

best_lightgbm_probabilities = (
    best_lightgbm_model.predict_proba(
        X_validation_preprocessed
    )[:, 1]
)

best_lightgbm_predictions = (
    best_lightgbm_probabilities >= 0.50
).astype(int)

best_lightgbm_results = evaluate_binary_classifier(
    model_name="Best Tuned LightGBM",
    y_true=y_validation,
    y_pred=best_lightgbm_predictions,
    y_probability=best_lightgbm_probabilities
)

# 6. Display compact tuning comparison

compact_columns = [
    "model",
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall_sensitivity",
    "specificity",
    "f1_score",
    "roc_auc",
    "pr_auc",
    "false_positives",
    "false_negatives",
    "true_positives",
]

compact_lgbm_results = lightgbm_tuning_results[
    compact_columns
].copy()

percentage_columns = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall_sensitivity",
    "specificity",
    "f1_score",
    "roc_auc",
    "pr_auc",
]

compact_lgbm_results[percentage_columns] = (
    compact_lgbm_results[percentage_columns] * 100
)

print("\nLightGBM tuning completed successfully.")
print("-" * 100)

print("\nCompact LightGBM tuning results:")
print(
    compact_lgbm_results
    .round(2)
    .to_string(index=False)
)

print("\nBest LightGBM configuration:")
print(best_lightgbm_config_name)

print("\nBest LightGBM selected metrics:")
print(
    best_lightgbm_results[
        [
            "accuracy",
            "balanced_accuracy",
            "precision",
            "recall_sensitivity",
            "specificity",
            "f1_score",
            "roc_auc",
            "pr_auc",
            "false_positives",
            "false_negatives",
            "true_positives",
        ]
    ]
    .round(4)
    .T
    .to_string(header=False)
)

print("\nBest LightGBM parameters:")
parameter_columns = [
    "n_estimators",
    "learning_rate",
    "num_leaves",
    "max_depth",
    "min_child_samples",
    "subsample",
    "colsample_bytree",
    "reg_lambda",
    "reg_alpha",
    "scale_pos_weight",
]

for parameter in parameter_columns:
    print(f"{parameter}: {best_lightgbm_row[parameter]}")

print("\nTest set status:")
print("- Test data remains untouched.")
print("- LightGBM used training and validation data only.")

Training class distribution:
Negative class count: 61,531
Positive class count: 7,936
Base scale_pos_weight: 7.7534

Training LGBM_01_Balanced_Reference...
scale_pos_weight: 7.7534
Completed LGBM_01_Balanced_Reference in 0.43 seconds.

Training LGBM_02_More_Regularized...
scale_pos_weight: 7.7534
Completed LGBM_02_More_Regularized in 0.66 seconds.

Training LGBM_03_Deeper_Trees...
scale_pos_weight: 7.7534
Completed LGBM_03_Deeper_Trees in 0.50 seconds.

Training LGBM_04_Lower_Positive_Weight...
scale_pos_weight: 6.5904
Completed LGBM_04_Lower_Positive_Weight in 0.47 seconds.

Training LGBM_05_Higher_Positive_Weight...
scale_pos_weight: 8.9164
Completed LGBM_05_Higher_Positive_Weight in 0.45 seconds.

LightGBM tuning completed successfully.
----------------------------------------------------------------------------------------------------

Compact LightGBM tuning results:
                         model  accuracy  balanced_accuracy  precision  recall_sensitivity  specificity  f1_score  

In [4]:
# Compact LightGBM tuning results

compact_columns = [
    "model",
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall_sensitivity",
    "specificity",
    "f1_score",
    "roc_auc",
    "pr_auc",
    "false_positives",
    "false_negatives",
    "true_positives",
]

compact_lgbm_results = lightgbm_tuning_results[
    compact_columns
].copy()

percentage_columns = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall_sensitivity",
    "specificity",
    "f1_score",
    "roc_auc",
    "pr_auc",
]

compact_lgbm_results[percentage_columns] = (
    compact_lgbm_results[percentage_columns] * 100
)

print("Compact LightGBM Tuning Results")
print("-" * 110)

print(
    compact_lgbm_results
    .round(2)
    .to_string(index=False)
)

print("\nBest LightGBM configuration:")
print(best_lightgbm_config_name)

print("\nBest LightGBM selected metrics:")
print(
    best_lightgbm_results[
        [
            "accuracy",
            "balanced_accuracy",
            "precision",
            "recall_sensitivity",
            "specificity",
            "f1_score",
            "roc_auc",
            "pr_auc",
            "false_positives",
            "false_negatives",
            "true_positives",
        ]
    ]
    .round(4)
    .T
    .to_string(header=False)
)

Compact LightGBM Tuning Results
--------------------------------------------------------------------------------------------------------------
                         model  accuracy  balanced_accuracy  precision  recall_sensitivity  specificity  f1_score  roc_auc  pr_auc  false_positives  false_negatives  true_positives
          LGBM_03_Deeper_Trees     65.95              63.07      18.50               59.35        66.79     28.21    67.85   24.04             4391              683             997
      LGBM_02_More_Regularized     64.87              62.98      18.20               60.54        65.42     27.98    67.87   23.79             4572              663            1017
 LGBM_04_Lower_Positive_Weight     73.07              62.87      20.86               49.70        76.04     29.39    67.98   24.12             3167              845             835
    LGBM_01_Balanced_Reference     65.22              62.86      18.23               59.82        65.91     27.95    68.06   23.91   

In [5]:
# Step 4: Build and evaluate soft voting ensembles

import joblib
import numpy as np
import pandas as pd

from scipy import sparse
from catboost import CatBoostClassifier, Pool

# 1. Load best saved Notebook 6 models

MODELS_PATH = PROJECT_ROOT / "models"

BEST_XGBOOST_PATH = (
    MODELS_PATH
    / "notebook_6_best_tuned_xgboost.joblib"
)

BEST_CATBOOST_PATH = (
    MODELS_PATH
    / "notebook_6_best_tuned_catboost.cbm"
)

BEST_HGB_PATH = (
    MODELS_PATH
    / "notebook_6_best_tuned_hist_gradient_boosting.joblib"
)

if not BEST_XGBOOST_PATH.exists():
    raise FileNotFoundError(BEST_XGBOOST_PATH)

if not BEST_CATBOOST_PATH.exists():
    raise FileNotFoundError(BEST_CATBOOST_PATH)

if not BEST_HGB_PATH.exists():
    raise FileNotFoundError(BEST_HGB_PATH)

best_xgboost_model = joblib.load(BEST_XGBOOST_PATH)

best_catboost_model = CatBoostClassifier()
best_catboost_model.load_model(BEST_CATBOOST_PATH)

best_hgb_model = joblib.load(BEST_HGB_PATH)

# best_lightgbm_model already exists from Step 3

# 2. Prepare validation inputs for each model type

# XGBoost and LightGBM use preprocessed 179-feature matrix
X_validation_xgb_lgbm = X_validation_preprocessed

# HistGradientBoosting requires dense input
if sparse.issparse(X_validation_preprocessed):
    X_validation_hgb = (
        X_validation_preprocessed
        .toarray()
        .astype(np.float32)
    )
else:
    X_validation_hgb = (
        np.asarray(X_validation_preprocessed)
        .astype(np.float32)
    )

# CatBoost uses raw 43 predictors with native categorical handling
X_validation_catboost = X_validation.copy()

for column in CATEGORICAL_FEATURES:
    X_validation_catboost[column] = (
        X_validation_catboost[column]
        .astype(str)
    )

for column in NUMERIC_FEATURES:
    X_validation_catboost[column] = pd.to_numeric(
        X_validation_catboost[column],
        errors="raise"
    )

validation_pool = Pool(
    data=X_validation_catboost,
    label=y_validation,
    cat_features=CATEGORICAL_FEATURES
)

# 3. Generate validation probabilities

xgb_probabilities = (
    best_xgboost_model
    .predict_proba(X_validation_xgb_lgbm)[:, 1]
)

cat_probabilities = (
    best_catboost_model
    .predict_proba(validation_pool)[:, 1]
)

hgb_probabilities = (
    best_hgb_model
    .predict_proba(X_validation_hgb)[:, 1]
)

lgbm_probabilities = (
    best_lightgbm_model
    .predict_proba(X_validation_xgb_lgbm)[:, 1]
)

# 4. Define ensemble weighting strategies

ensemble_probability_configs = {
    "Ensemble_01_Equal_XGB_CAT_HGB": (
        xgb_probabilities
        + cat_probabilities
        + hgb_probabilities
    ) / 3,

    "Ensemble_02_Equal_XGB_CAT_HGB_LGBM": (
        xgb_probabilities
        + cat_probabilities
        + hgb_probabilities
        + lgbm_probabilities
    ) / 4,

    "Ensemble_03_Weighted_XGB_CAT_HGB": (
        0.50 * xgb_probabilities
        + 0.30 * cat_probabilities
        + 0.20 * hgb_probabilities
    ),

    "Ensemble_04_Weighted_XGB_CAT_LGBM": (
        0.50 * xgb_probabilities
        + 0.25 * cat_probabilities
        + 0.25 * lgbm_probabilities
    ),

    "Ensemble_05_Weighted_All_Advanced": (
        0.40 * xgb_probabilities
        + 0.25 * cat_probabilities
        + 0.20 * hgb_probabilities
        + 0.15 * lgbm_probabilities
    ),

    "Ensemble_06_Recall_Weighted_All": (
        0.30 * xgb_probabilities
        + 0.25 * cat_probabilities
        + 0.20 * hgb_probabilities
        + 0.25 * lgbm_probabilities
    ),
}

# 5. Evaluate ensembles at threshold 0.50

ensemble_results = []

for ensemble_name, probabilities in ensemble_probability_configs.items():

    predictions = (
        probabilities >= 0.50
    ).astype(int)

    result = evaluate_binary_classifier(
        model_name=ensemble_name,
        y_true=y_validation,
        y_pred=predictions,
        y_probability=probabilities
    )

    ensemble_results.append(result)

ensemble_results = pd.concat(
    ensemble_results,
    ignore_index=True
)

ensemble_results = (
    ensemble_results
    .sort_values(
        by=[
            "balanced_accuracy",
            "f1_score",
            "roc_auc",
            "pr_auc",
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

# 6. Compare with current best Notebook 6 XGBoost and LightGBM

notebook_6_selected_result = pd.DataFrame(
    [
        {
            "model": "Notebook 6 Selected XGBoost",
            "accuracy": notebook_6_metadata["selected_candidate"]["accuracy"],
            "balanced_accuracy": notebook_6_metadata["selected_candidate"]["balanced_accuracy"],
            "precision": notebook_6_metadata["selected_candidate"]["precision"],
            "recall_sensitivity": notebook_6_metadata["selected_candidate"]["recall_sensitivity"],
            "specificity": notebook_6_metadata["selected_candidate"]["specificity"],
            "f1_score": notebook_6_metadata["selected_candidate"]["f1_score"],
            "roc_auc": notebook_6_metadata["selected_candidate"]["roc_auc"],
            "pr_auc": notebook_6_metadata["selected_candidate"]["pr_auc"],
            "predicted_positive_rate": notebook_6_metadata["selected_candidate"]["predicted_positive_rate"],
            "true_negatives": notebook_6_metadata["selected_candidate"]["true_negatives"],
            "false_positives": notebook_6_metadata["selected_candidate"]["false_positives"],
            "false_negatives": notebook_6_metadata["selected_candidate"]["false_negatives"],
            "true_positives": notebook_6_metadata["selected_candidate"]["true_positives"],
        }
    ]
)

advanced_model_comparison = pd.concat(
    [
        notebook_6_selected_result,
        best_lightgbm_results.assign(
            model="Best Tuned LightGBM"
        ),
        ensemble_results,
    ],
    ignore_index=True
)

advanced_model_comparison = (
    advanced_model_comparison
    .sort_values(
        by=[
            "balanced_accuracy",
            "f1_score",
            "roc_auc",
            "pr_auc",
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

# 
# 7. compact comparison

display_columns = [
    "model",
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall_sensitivity",
    "specificity",
    "f1_score",
    "roc_auc",
    "pr_auc",
    "false_positives",
    "false_negatives",
    "true_positives",
]

display_advanced_comparison = advanced_model_comparison[
    display_columns
].copy()

percentage_columns = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall_sensitivity",
    "specificity",
    "f1_score",
    "roc_auc",
    "pr_auc",
]

display_advanced_comparison[percentage_columns] = (
    display_advanced_comparison[percentage_columns] * 100
)

print("Notebook 6B Advanced Model Comparison — Validation Set")
print("-" * 115)

print(
    display_advanced_comparison
    .round(2)
    .to_string(index=False)
)

# 8. Select best advanced candidate

best_advanced_row = advanced_model_comparison.iloc[0]
best_advanced_model_name = best_advanced_row["model"]

print("\nBest advanced validation candidate:")
print(best_advanced_model_name)

print("\nBest advanced candidate selected metrics:")
print(
    best_advanced_row[
        [
            "accuracy",
            "balanced_accuracy",
            "precision",
            "recall_sensitivity",
            "specificity",
            "f1_score",
            "roc_auc",
            "pr_auc",
            "false_positives",
            "false_negatives",
            "true_positives",
        ]
    ]
    .round(4)
    .to_string()
)

print("\nTest set status:")
print("- Test data remains untouched.")
print("- Ensemble comparison used validation data only.")

Notebook 6B Advanced Model Comparison — Validation Set
-------------------------------------------------------------------------------------------------------------------
                             model  accuracy  balanced_accuracy  precision  recall_sensitivity  specificity  f1_score  roc_auc  pr_auc  false_positives  false_negatives  true_positives
 Ensemble_05_Weighted_All_Advanced     66.53              63.19      18.71               58.87        67.50     28.40    68.19   24.39             4296              691             989
  Ensemble_03_Weighted_XGB_CAT_HGB     66.61              63.18      18.73               58.75        67.61     28.41    68.20   24.36             4282              693             987
       Notebook 6 Selected XGBoost     66.56              63.15      18.70               58.75        67.55     28.37    68.07   24.11             4290              693             987
Ensemble_02_Equal_XGB_CAT_HGB_LGBM     66.45              63.12      18.66               

In [ ]:
# Step 5: Save advanced modeling results and selected ensemble

import json
import joblib
from datetime import datetime

# 1. Create output directories

MODELS_PATH = PROJECT_ROOT / "models"
ARTIFACTS_PATH = PROJECT_ROOT / "artifacts"
METRICS_PATH = PROJECT_ROOT / "outputs" / "metrics"

MODELS_PATH.mkdir(parents=True, exist_ok=True)
ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)
METRICS_PATH.mkdir(parents=True, exist_ok=True)

# 2. Define save paths

lightgbm_results_save_path = (
    METRICS_PATH
    / "notebook_6b_lightgbm_tuning_results.csv"
)

advanced_comparison_save_path = (
    METRICS_PATH
    / "notebook_6b_advanced_model_comparison.csv"
)

metadata_save_path = (
    ARTIFACTS_PATH
    / "notebook_6b_advanced_modeling_metadata.json"
)

completion_summary_path = (
    METRICS_PATH
    / "notebook_6b_completion_summary.csv"
)

best_lightgbm_model_path = (
    MODELS_PATH
    / "notebook_6b_best_tuned_lightgbm.joblib"
)

selected_ensemble_package_path = (
    MODELS_PATH
    / "notebook_6b_selected_advanced_ensemble.joblib"
)

# 3. Save result tables

lightgbm_tuning_results.to_csv(
    lightgbm_results_save_path,
    index=False
)

advanced_model_comparison.to_csv(
    advanced_comparison_save_path,
    index=False
)

# 4. Save best LightGBM model

joblib.dump(
    best_lightgbm_model,
    best_lightgbm_model_path
)

# 5. Define ensemble weights used by selected advanced model

selected_ensemble_weights = {
    "xgboost": 0.40,
    "catboost": 0.25,
    "hist_gradient_boosting": 0.20,
    "lightgbm": 0.15,
}

selected_ensemble_components = {
    "xgboost_model": best_xgboost_model,
    "catboost_model": best_catboost_model,
    "hist_gradient_boosting_model": best_hgb_model,
    "lightgbm_model": best_lightgbm_model,
}

selected_ensemble_package = {
    "selected_model_name": best_advanced_model_name,
    "selected_model_type": "Weighted soft voting ensemble",
    "selected_threshold": 0.50,
    "weights": selected_ensemble_weights,
    "components": selected_ensemble_components,
    "input_requirements": {
        "xgboost": "preprocessed 179-feature matrix",
        "lightgbm": "preprocessed 179-feature matrix",
        "hist_gradient_boosting": "dense preprocessed 179-feature matrix",
        "catboost": "raw 43 predictors with categorical columns as strings",
    },
}

joblib.dump(
    selected_ensemble_package,
    selected_ensemble_package_path
)

# 6. Create metadata

selected_advanced_summary = {
    "selected_model_name": best_advanced_model_name,
    "selected_model_type": "Weighted soft voting ensemble",
    "selected_threshold": 0.50,
    "accuracy": float(best_advanced_row["accuracy"]),
    "balanced_accuracy": float(best_advanced_row["balanced_accuracy"]),
    "precision": float(best_advanced_row["precision"]),
    "recall_sensitivity": float(best_advanced_row["recall_sensitivity"]),
    "specificity": float(best_advanced_row["specificity"]),
    "f1_score": float(best_advanced_row["f1_score"]),
    "roc_auc": float(best_advanced_row["roc_auc"]),
    "pr_auc": float(best_advanced_row["pr_auc"]),
    "false_positives": int(best_advanced_row["false_positives"]),
    "false_negatives": int(best_advanced_row["false_negatives"]),
    "true_positives": int(best_advanced_row["true_positives"]),
    "true_negatives": int(best_advanced_row["true_negatives"]),
}

notebook_6b_metadata = {
    "notebook": "06B_advanced_modeling.ipynb",
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "purpose": (
        "Optional advanced modeling after Notebook 6 using LightGBM "
        "and weighted soft voting ensembles."
    ),
    "target_column": TARGET_COLUMN,
    "group_column": GROUP_COLUMN,
    "raw_feature_count": len(MODEL_FEATURES),
    "train_encounters": int(len(train_df)),
    "validation_encounters": int(len(validation_df)),
    "reserved_test_encounters": int(len(test_df)),
    "test_set_used": False,
    "advanced_models_tested": [
        "LightGBM",
        "Soft voting ensembles",
    ],
    "best_lightgbm_config": best_lightgbm_config_name,
    "selected_advanced_candidate": selected_advanced_summary,
    "ensemble_weights": selected_ensemble_weights,
    "saved_files": {
        "lightgbm_tuning_results": str(lightgbm_results_save_path),
        "advanced_model_comparison": str(advanced_comparison_save_path),
        "best_lightgbm_model": str(best_lightgbm_model_path),
        "selected_advanced_ensemble": str(selected_ensemble_package_path),
        "metadata": str(metadata_save_path),
        "completion_summary": str(completion_summary_path),
    },
}

with open(
    metadata_save_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        notebook_6b_metadata,
        file,
        indent=4
    )

# 7. Save completion summary

completion_summary = pd.DataFrame(
    [
        {
            "notebook": "06B_advanced_modeling.ipynb",
            "status": "Complete",
            "selected_model_name": best_advanced_model_name,
            "selected_model_type": "Weighted soft voting ensemble",
            "selected_threshold": 0.50,
            "selected_accuracy": float(best_advanced_row["accuracy"]),
            "selected_balanced_accuracy": float(
                best_advanced_row["balanced_accuracy"]
            ),
            "selected_precision": float(best_advanced_row["precision"]),
            "selected_recall": float(
                best_advanced_row["recall_sensitivity"]
            ),
            "selected_specificity": float(best_advanced_row["specificity"]),
            "selected_f1_score": float(best_advanced_row["f1_score"]),
            "selected_roc_auc": float(best_advanced_row["roc_auc"]),
            "selected_pr_auc": float(best_advanced_row["pr_auc"]),
            "test_set_used": False,
        }
    ]
)

completion_summary.to_csv(
    completion_summary_path,
    index=False
)

# 8. Confirm files exist

saved_paths = [
    lightgbm_results_save_path,
    advanced_comparison_save_path,
    metadata_save_path,
    completion_summary_path,
    best_lightgbm_model_path,
    selected_ensemble_package_path,
]

missing_saved_files = [
    str(path)
    for path in saved_paths
    if not path.exists()
]

if missing_saved_files:
    raise FileNotFoundError(
        "Some Notebook 6B outputs were not saved:\n"
        + "\n".join(missing_saved_files)
    )

# 9. Display confirmation

print("Notebook 6B artifacts saved successfully.")
print("-" * 80)

print("\nSelected advanced validation candidate:")
print(f"Model: {best_advanced_model_name}")
print("Model type: Weighted soft voting ensemble")
print("Selected threshold: 0.50")

print("\nSelected advanced validation results:")
print(f"Accuracy: {best_advanced_row['accuracy']:.4f}")
print(f"Balanced accuracy: {best_advanced_row['balanced_accuracy']:.4f}")
print(f"Precision: {best_advanced_row['precision']:.4f}")
print(f"Recall: {best_advanced_row['recall_sensitivity']:.4f}")
print(f"Specificity: {best_advanced_row['specificity']:.4f}")
print(f"F1 score: {best_advanced_row['f1_score']:.4f}")
print(f"ROC-AUC: {best_advanced_row['roc_auc']:.4f}")
print(f"PR-AUC: {best_advanced_row['pr_auc']:.4f}")
print(f"False positives: {int(best_advanced_row['false_positives'])}")
print(f"False negatives: {int(best_advanced_row['false_negatives'])}")
print(f"True positives: {int(best_advanced_row['true_positives'])}")

print("\nSaved files:")
print(f"- {advanced_comparison_save_path}")
print(f"- {lightgbm_results_save_path}")
print(f"- {best_lightgbm_model_path}")
print(f"- {selected_ensemble_package_path}")
print(f"- {metadata_save_path}")
print(f"- {completion_summary_path}")

print("\nNotebook 6B status:")
print("- LightGBM testing completed.")
print("- Soft voting ensemble testing completed.")
print("- Advanced ensemble saved.")
print("- Test set remains completely untouched.")

Notebook 6B artifacts saved successfully.
--------------------------------------------------------------------------------

Selected advanced validation candidate:
Model: Ensemble_05_Weighted_All_Advanced
Model type: Weighted soft voting ensemble
Selected threshold: 0.50

Selected advanced validation results:
Accuracy: 0.6653
Balanced accuracy: 0.6319
Precision: 0.1871
Recall: 0.5887
Specificity: 0.6750
F1 score: 0.2840
ROC-AUC: 0.6819
PR-AUC: 0.2439
False positives: 4296
False negatives: 691
True positives: 989

Saved files:
- C:\Users\pradh\Documents\hospital-readmission-project\outputs\metrics\notebook_6b_advanced_model_comparison.csv
- C:\Users\pradh\Documents\hospital-readmission-project\outputs\metrics\notebook_6b_lightgbm_tuning_results.csv
- C:\Users\pradh\Documents\hospital-readmission-project\models\notebook_6b_best_tuned_lightgbm.joblib
- C:\Users\pradh\Documents\hospital-readmission-project\models\notebook_6b_selected_advanced_ensemble.joblib
- C:\Users\pradh\Documents\hosp